In [1]:
import nest_asyncio
nest_asyncio.apply()

import gradio as gr

import pandas as pd
import os

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from agent_framework import Agent
from agent_framework import GroupChatBuilder, GroupChatStateSnapshot
from agent_framework.events import AgentRunUpdateEvent, WorkflowOutputEvent


c:\Users\luisg\Desktop\js_anthony\Coches_segunda_mano_ia\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: module 'mcp.types' has no attribute 'ToolUseContent'

In [ ]:
CSV_PATH = "/data/autofesa_completo_20251202_0932.csv"

def cargar_docs():
    if not os.path.exists(CSV_PATH):
        return [
            Document(page_content="BMW 320d Blanco", metadata={"Modelo":"BMW 320d","Precio":20000,"Km":50000,"Link":"http://auto/1"}),
            Document(page_content="Audi A4 Negro", metadata={"Modelo":"Audi A4","Precio":15000,"Km":80000,"Link":"http://auto/2"}),
        ]

    df = pd.read_csv(CSV_PATH)
    docs = []
    for _, row in df.iterrows():
        texto = f"{row['Modelo']} {row['Combustible']} Año {row['Año']}"
        meta = {
            "Modelo": row["Modelo"],
            "Precio": int(row["Precio"]),
            "Km": int(row["Km"]),
            "Link": row["Link"]
        }
        docs.append(Document(page_content=texto, metadata=meta))
    return docs

docs = cargar_docs()
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, emb)
s